# architecture extension contract v1

目标：把“稳定 `src/` surface 后如需继续扩展，必须说明原因”固化为 `src.core` 契约，而不是依赖聊天记录或临时说明。

边界：notebook 只编排证据；决策 hash、registry 和 review 写出逻辑必须在 `src.core`。

In [1]:
from pathlib import Path
import json
import os
import sys

REPO_ROOT = Path.cwd().resolve()
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').exists() and (candidate / 'AGENTS.md').exists():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import (
    architecture_extension_content_hash,
    build_architecture_extension_decision,
    write_architecture_extension_decision,
)

CARD = Path('research/architecture_extension_contract_v1')
CARD.mkdir(parents=True, exist_ok=True)
CARD


PosixPath('research/architecture_extension_contract_v1')

In [2]:
decision = build_architecture_extension_decision(
    decision_id='src-core-architecture-extension-contract-v1',
    target_surface='src/core',
    change_type='governance_contract',
    reason='原始架构要求稳定 src 后继续扩展必须说明原因；需要把原因、影响范围、证据 notebook 和验证命令变成可审阅契约。',
    impact_scope=('src/core', 'tests/test_core_architecture_decisions.py', 'research/architecture_extension_contract_v1'),
    evidence_notebook='research/architecture_extension_contract_v1/architecture_extension_contract_v1.ipynb',
    verification_commands=(
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests/test_core_architecture_decisions.py -q',
        'PYTHONPATH=. /Users/rziiiii/dev/workspace/qlib_venv/bin/python -m pytest tests -q',
    ),
    alternatives_considered=(
        '只在 AGENTS.md 中描述扩展原因要求',
        '把扩展原因写入 skill',
        '只保留聊天记录说明',
    ),
    constraints=(
        'skill 只能承载方法论，不保存项目状态',
        'research/production 必须 notebook-first 并有中文结论',
        '稳定 src 扩展必须有聚焦测试和可重算 hash',
    ),
    created_at='2026-06-19T03:30:00Z',
)
decision


{'alternatives_considered': ['只在 AGENTS.md 中描述扩展原因要求',
  '把扩展原因写入 skill',
  '只保留聊天记录说明'],
 'blocked_without_evidence': True,
 'change_type': 'governance_contract',
 'constraints': ['skill 只能承载方法论，不保存项目状态',
  'research/production 必须 notebook-first 并有中文结论',
  '稳定 src 扩展必须有聚焦测试和可重算 hash'],
 'created_at': '2026-06-19T03:30:00Z',
 'decision_hash': 'ec3bb692d7cd7775daa63cde9951535bc703a22bddf52e60cb5c8dd718a596e9',
 'decision_id': 'src-core-architecture-extension-contract-v1',
 'evidence_notebook': 'research/architecture_extension_contract_v1/architecture_extension_contract_v1.ipynb',
 'extension_requires_reason': True,
 'impact_scope': ['src/core',
  'tests/test_core_architecture_decisions.py',
  'research/architecture_extension_contract_v1'],
 'reason': '原始架构要求稳定 src 后继续扩展必须说明原因；需要把原因、影响范围、证据 notebook 和验证命令变成可审阅契约。',
 'review_gate': 'src extension requires reason, evidence notebook, focused verification, and Chinese review.',
 'schema_version': 'core.architecture_extension_decision.v1',
 '

In [3]:
write_result = write_architecture_extension_decision(
    output_dir=CARD,
    decision=decision,
    registry_path=CARD / 'architecture_extension_registry.json',
)
decision_payload = json.loads(Path(write_result['decision_path']).read_text(encoding='utf-8'))
registry = json.loads(Path(write_result['registry_path']).read_text(encoding='utf-8'))
review_text = Path(write_result['review_path']).read_text(encoding='utf-8')
write_result


{'schema_version': 'core.architecture_extension_write_result.v1',
 'decision_path': 'research/architecture_extension_contract_v1/src-core-architecture-extension-contract-v1.json',
 'registry_path': 'research/architecture_extension_contract_v1/architecture_extension_registry.json',
 'review_path': 'research/architecture_extension_contract_v1/architecture_extension_review.md',
 'decision_hash': 'ec3bb692d7cd7775daa63cde9951535bc703a22bddf52e60cb5c8dd718a596e9',
 'registry_hash': 'fb552a57250bc26d10a28ca2de5792efa16856e37c65a267cb231234a348b26c'}

In [4]:
summary = {
    'schema_version': 'research.architecture_extension_contract.v1',
    'status': 'completed',
    'decision_id': decision_payload['decision_id'],
    'target_surface': decision_payload['target_surface'],
    'change_type': decision_payload['change_type'],
    'extension_requires_reason': decision_payload['extension_requires_reason'],
    'blocked_without_evidence': decision_payload['blocked_without_evidence'],
    'skill_boundary': decision_payload['skill_boundary'],
    'decision_hash': decision_payload['decision_hash'],
    'computed_decision_hash': architecture_extension_content_hash(decision_payload),
    'registry_hash': registry['registry_hash'],
    'registry_decision_count': len(registry['decisions']),
    'artifacts': {
        'decision': Path(write_result['decision_path']).name,
        'registry': Path(write_result['registry_path']).name,
        'review': Path(write_result['review_path']).name,
    },
}

assert decision_payload['schema_version'] == 'core.architecture_extension_decision.v1'
assert decision_payload['decision_hash'] == summary['computed_decision_hash']
assert decision_payload['extension_requires_reason'] is True
assert decision_payload['blocked_without_evidence'] is True
assert decision_payload['skill_boundary'] == 'method_only'
assert registry['schema_version'] == 'core.architecture_extension_registry.v1'
assert registry['decisions'][decision_payload['decision_id']]['decision_hash'] == decision_payload['decision_hash']
assert '中文结论' in review_text
assert '扩展原因' in review_text

(CARD / 'analysis_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


{'schema_version': 'research.architecture_extension_contract.v1',
 'status': 'completed',
 'decision_id': 'src-core-architecture-extension-contract-v1',
 'target_surface': 'src/core',
 'change_type': 'governance_contract',
 'extension_requires_reason': True,
 'blocked_without_evidence': True,
 'skill_boundary': 'method_only',
 'decision_hash': 'ec3bb692d7cd7775daa63cde9951535bc703a22bddf52e60cb5c8dd718a596e9',
 'computed_decision_hash': 'ec3bb692d7cd7775daa63cde9951535bc703a22bddf52e60cb5c8dd718a596e9',
 'registry_hash': 'fb552a57250bc26d10a28ca2de5792efa16856e37c65a267cb231234a348b26c',
 'registry_decision_count': 1,
 'artifacts': {'decision': 'src-core-architecture-extension-contract-v1.json',
  'registry': 'architecture_extension_registry.json',
  'review': 'architecture_extension_review.md'}}

## 中文结论

- 做了什么：新增并验证 `src.core` 架构扩展决策契约，用 `build_architecture_extension_decision`、`architecture_extension_content_hash` 和 `write_architecture_extension_decision` 记录稳定 `src/` surface 继续扩展时的原因、影响范围、证据 notebook、验证命令、registry 和中文 review。
- 核心指标：`extension_requires_reason=True`，`blocked_without_evidence=True`，`skill_boundary=method_only`，decision hash 可以从内容重算，registry 记录 1 条决策并带 `registry_hash`。
- regime 差异：本实验不训练模型，也不重新计算 market regime；它补的是平台治理契约，保证后续 `src/analysis`、`src/model`、`src/factor` 等 regime/model 相关扩展必须留下原因和证据。
- 是否晋升：否。本 notebook 只验证架构契约，不晋升任何交易策略或模型。
- 风险：该契约只能约束新增记录的格式和 hash，不能自动判断架构理由是否充分；review 时仍需要结合代码 diff、notebook 证据和测试覆盖人工判断。
- 下一步：后续新增稳定 `src/` surface 或改变跨模块契约时，应把对应原因写入 architecture extension registry，并在 dashboard/schedule/production 证据链中引用。